# Runbook 2.2.b: Environmental vulnerabilities (power, HVAC, fire, water)  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 2.2.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A data center streams readings from environmental sensors placed around each server room:
power (voltage as a percent of nominal), temperature (Celsius), humidity (percent relative
humidity), smoke detectors, and water/flood detectors. Facilities staff need every reading
automatically triaged into a severity level so a human only has to look at what matters.

Your task is to write a function `solve(data)` that takes a list of sensor readings and returns
a dictionary mapping each reading's `id` to a severity string: `"NORMAL"`, `"WARNING"`, or
`"CRITICAL"`.

### Per-reading thresholds (by `type`)

**`power`** (`value` = supply voltage as a percent of nominal, 100 = nominal):
- `CRITICAL` if `value == 0` (total outage) or `value < 90` or `value > 110`
- `WARNING` if `90 <= value < 95` or `105 < value <= 110`
- `NORMAL` otherwise (`95 <= value <= 105`)

**`temperature`** (`value` = degrees Celsius):
- `CRITICAL` if `value > 30`
- `WARNING` if `27 < value <= 30` or `value < 15`
- `NORMAL` otherwise (`15 <= value <= 27`)

**`humidity`** (`value` = percent relative humidity):
- `CRITICAL` if `value > 70` or `value < 20`
- `WARNING` if `60 < value <= 70` or `20 <= value < 40`
- `NORMAL` otherwise (`40 <= value <= 60`)

**`smoke`** (`value` = detector reading, `0` = clear, any positive value = smoke detected):
- `CRITICAL` if `value > 0`
- `NORMAL` otherwise

**`water`** (`value` = detector reading, `0` = dry, any positive value = moisture detected):
- `CRITICAL` if `value > 0`
- `NORMAL` otherwise

### Combined-signal escalation (cooling-failure pattern)
After computing each reading's per-reading severity above, apply one escalation rule: for
readings that share the same `room`, if there is **both** a `temperature` reading rated
`WARNING` or `CRITICAL` **and** a `humidity` reading rated `WARNING` or `CRITICAL` in that same
room, escalate *both* of those specific readings (temperature and humidity only) to
`CRITICAL`. This models a failing HVAC unit, where rising temperature and drifting humidity
together are a stronger signal than either alone. Other reading types in the room (power, smoke,
water) are not affected by this rule, and a room where only one of the two (temperature or
humidity) is elevated is not escalated.

**Lesson objective.** Identify how power loss, HVAC (heating, ventilation, and air conditioning) failure, fire, and water/flood events threaten the availability and integrity of physical IT infrastructure, and you can evaluate whether a facility's environmental sensor readings indicate a developing incident.

### What comes in
A list of dictionaries, each representing one sensor reading:
```python
[
    {"id": "s1", "room": "server-room-a", "type": "temperature", "value": 29},
    {"id": "s2", "room": "server-room-a", "type": "humidity", "value": 65},
    {"id": "s3", "room": "server-room-b", "type": "power", "value": 100},
]
```

### What must come out
A dictionary mapping each reading's `"id"` to its severity string:
```python
{
    "s1": "CRITICAL",   # escalated: room also had a humidity WARNING/CRITICAL
    "s2": "CRITICAL",   # escalated: room also had a temperature WARNING/CRITICAL
    "s3": "NORMAL",
}
```

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.


### Rules to respect
- [ ] `type` is one of `"power"`, `"temperature"`, `"humidity"`, `"smoke"`, `"water"`.
- [ ] `value` is a non-negative number (float or int) except `power`, which may be `0` (outage).
- [ ] Pure function, deterministic, standard library only.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(data: list[dict]) -> dict[str, str]
```

#### Input (`data`)
A list of dictionaries representing sensor readings. Each dictionary contains:
- `id` (str): Unique identifier for the reading.
- `room` (str): Room/zone identifier the sensor is located in.
- `type` (str): One of `"power"`, `"temperature"`, `"humidity"`, `"smoke"`, `"water"`.
- `value` (float/int): The sensor's numeric reading (see the problem statement above for units per type).

#### Output (`result`)
A dictionary where keys are the `id` of each reading from the input and values are the severity
string: `"NORMAL"`, `"WARNING"`, or `"CRITICAL"`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Per-type thresholds (see the problem statement above for full detail):
  1. `power`: outage (`0`) or `<90` or `>110` -> `CRITICAL`; `[90,95)` or `(105,110]` -> `WARNING`; else `NORMAL`.
  2. `temperature`: `>30` -> `CRITICAL`; `(27,30]` or `<15` -> `WARNING`; else `NORMAL`.
  3. `humidity`: `>70` or `<20` -> `CRITICAL`; `(60,70]` or `[20,40)` -> `WARNING`; else `NORMAL`.
  4. `smoke`/`water`: any positive value -> `CRITICAL`; `0` -> `NORMAL`.
  5. Combined-signal escalation: within the same `room`, if a `temperature` reading is
     `WARNING`/`CRITICAL` AND a `humidity` reading is `WARNING`/`CRITICAL`, escalate both of
     those two readings to `CRITICAL` (only temperature and humidity readings are affected).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
def _power_severity(value: float) -> str:
    if value == 0 or value < 90 or value > 110:
        return "CRITICAL"
    if (90 <= value < 95) or (105 < value <= 110):
        return "WARNING"
    return "NORMAL"


def _temperature_severity(value: float) -> str:
    if value > 30:
        return "CRITICAL"
    if (27 < value <= 30) or value < 15:
        return "WARNING"
    return "NORMAL"


def _humidity_severity(value: float) -> str:
    if value > 70 or value < 20:
        return "CRITICAL"
    if (60 < value <= 70) or (20 <= value < 40):
        return "WARNING"
    return "NORMAL"


def _binary_severity(value: float) -> str:
    return "CRITICAL" if value > 0 else "NORMAL"


_RANK = {"NORMAL": 0, "WARNING": 1, "CRITICAL": 2}


def solve(data: list[dict]) -> dict[str, str]:
    severities: dict[str, str] = {}

    for reading in data:
        rid = reading["id"]
        rtype = reading["type"]
        value = reading["value"]

        if rtype == "power":
            severities[rid] = _power_severity(value)
        elif rtype == "temperature":
            severities[rid] = _temperature_severity(value)
        elif rtype == "humidity":
            severities[rid] = _humidity_severity(value)
        elif rtype in ("smoke", "water"):
            severities[rid] = _binary_severity(value)
        else:
            severities[rid] = "NORMAL"

    rooms: dict[str, dict[str, list[str]]] = {}
    for reading in data:
        room = reading["room"]
        rtype = reading["type"]
        if rtype in ("temperature", "humidity"):
            rooms.setdefault(room, {}).setdefault(rtype, []).append(reading["id"])

    for room, by_type in rooms.items():
        temp_ids = by_type.get("temperature", [])
        hum_ids = by_type.get("humidity", [])
        temp_elevated = any(_RANK[severities[i]] >= _RANK["WARNING"] for i in temp_ids)
        hum_elevated = any(_RANK[severities[i]] >= _RANK["WARNING"] for i in hum_ids)
        if temp_elevated and hum_elevated:
            for i in temp_ids:
                if _RANK[severities[i]] >= _RANK["WARNING"]:
                    severities[i] = "CRITICAL"
            for i in hum_ids:
                if _RANK[severities[i]] >= _RANK["WARNING"]:
                    severities[i] = "CRITICAL"

    return severities


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_power_levels():
    data = [
        {"id": "p1", "room": "a", "type": "power", "value": 0},
        {"id": "p2", "room": "a", "type": "power", "value": 92},
        {"id": "p3", "room": "a", "type": "power", "value": 100},
        {"id": "p4", "room": "a", "type": "power", "value": 108},
        {"id": "p5", "room": "a", "type": "power", "value": 120},
    ]
    assert impl.solve(data) == {
        "p1": "CRITICAL",
        "p2": "WARNING",
        "p3": "NORMAL",
        "p4": "WARNING",
        "p5": "CRITICAL",
    }


def test_temperature_and_humidity_no_escalation():
    data = [
        {"id": "t1", "room": "b", "type": "temperature", "value": 28},
        {"id": "h1", "room": "b", "type": "humidity", "value": 50},
    ]
    assert impl.solve(data) == {"t1": "WARNING", "h1": "NORMAL"}


def test_smoke_and_water():
    data = [
        {"id": "s1", "room": "c", "type": "smoke", "value": 0},
        {"id": "s2", "room": "c", "type": "smoke", "value": 3},
        {"id": "w1", "room": "c", "type": "water", "value": 0},
        {"id": "w2", "room": "c", "type": "water", "value": 1},
    ]
    assert impl.solve(data) == {
        "s1": "NORMAL",
        "s2": "CRITICAL",
        "w1": "NORMAL",
        "w2": "CRITICAL",
    }


def test_combined_escalation():
    data = [
        {"id": "t1", "room": "d", "type": "temperature", "value": 29},
        {"id": "h1", "room": "d", "type": "humidity", "value": 65},
        {"id": "p1", "room": "d", "type": "power", "value": 100},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL", "p1": "NORMAL"}


def test_accept():
    data = [
        {"id": "t1", "room": "e", "type": "temperature", "value": 20},
        {"id": "h1", "room": "e", "type": "humidity", "value": 45},
    ]
    assert impl.solve(data) == {"t1": "NORMAL", "h1": "NORMAL"}


def test_edge_empty():
    assert impl.solve([]) == {}


def test_multiple_rooms_independent():
    data = [
        {"id": "t1", "room": "room1", "type": "temperature", "value": 31},
        {"id": "h1", "room": "room2", "type": "humidity", "value": 75},
        {"id": "t2", "room": "room2", "type": "temperature", "value": 20},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL", "t2": "NORMAL"}


def test_already_critical_stays_critical_after_escalation():
    data = [
        {"id": "t1", "room": "f", "type": "temperature", "value": 35},
        {"id": "h1", "room": "f", "type": "humidity", "value": 65},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL"}


def test_low_temperature_and_low_humidity_escalate():
    data = [
        {"id": "t1", "room": "g", "type": "temperature", "value": 10},
        {"id": "h1", "room": "g", "type": "humidity", "value": 25},
    ]
    assert impl.solve(data) == {"t1": "CRITICAL", "h1": "CRITICAL"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     {}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it